# JoSho Trader — Full Prediction Loop
**Target: 97%+ accuracy per stock**

This notebook runs the complete prediction loop:
1. Download ALL 213 F&O stocks (full history)
2. Test ALL algorithms on ALL stocks
3. Save results + memory after each loop
4. Loop until 97%+ accuracy achieved

Run on Google Colab for free compute.

In [ ]:
# Cell 1: Setup
!git clone https://github.com/sijoyalmeida09/josho-trader.git 2>/dev/null; cd josho-trader && git pull
%cd josho-trader
!pip install -q xgboost lightgbm pandas numpy ta scikit-learn yfinance jugaad-data catboost

In [ ]:
# Cell 2: Download ALL 213 F&O stocks (full history)
!python download_all_fno.py

In [ ]:
# Cell 3: Run universal predictor on ALL stocks
!python -m src.ml.batch_scorer

In [ ]:
# Cell 4: View predictability ranking
import pandas as pd
df = pd.read_csv('data/results/stock_predictability_ranking.csv')
print(f'Total stocks scored: {len(df)}')
print(f'\nHIGHLY PREDICTABLE (score > 75):')
print(df[df['predictability_score'] > 75][['symbol','predictability_score','best_ml_accuracy','large_drop_winrate','recommendation']].to_string())
print(f'\nMODERATE (60-75):')
print(df[(df['predictability_score'] >= 60) & (df['predictability_score'] <= 75)][['symbol','predictability_score','best_ml_accuracy']].head(20).to_string())
print(f'\n97%+ Win Rate Stocks:')
high = df[df['large_drop_winrate'] >= 0.97]
if len(high) > 0:
    print(high[['symbol','large_drop_winrate','large_drop_trades']].to_string())
else:
    print('None yet — need more patterns or narrower conditions')

In [ ]:
# Cell 5: Run high win-rate pattern backtests on TOP stocks
!python run_high_winrate_backtest.py

In [ ]:
# Cell 6: Save results to Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
dst = '/content/drive/MyDrive/josho-trader-results'
os.makedirs(dst, exist_ok=True)
for f in ['data/results/stock_predictability_ranking.csv', 'data/results/loop2_results.json']:
    if os.path.exists(f): shutil.copy(f, dst)
shutil.copytree('data/models', f'{dst}/models', dirs_exist_ok=True)
print(f'Saved to {dst}')

In [ ]:
# Cell 7: Push results back to GitHub
!git add data/results/ data/models/ -f
!git commit -m 'results: Colab training run'
!git push